# Two-Stage Trust Verification (TSTV) for IoT-Cloud Security

Reproducible experimental implementation accompanying the manuscript **“Two-Stage Trust Verification for User and Data-Level Security in IoT-Cloud Networks.”**

This notebook implements the TSTV workflow and reproduces the performance, payload-size, cryptographic, and security-validation experiments reported in the study.

In [ ]:
# Reproducible experimental environment
import time
import hashlib
import hmac
import random
import statistics
import json
import pandas as pd
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

USER_SIZES = [100, 200, 300, 400, 500]

print("TSTV experimental environment initialized.")
print("User groups:", USER_SIZES)


## 1. User registration and UCP verification

In [ ]:
# ============================================================
# CELL 2 — TSTV USER REGISTRATION
# ============================================================

import hashlib
import secrets
import pandas as pd

def hash_password(password):
    """Create a SHA-256 hash for a user password."""
    return hashlib.sha256(password.encode()).hexdigest()


def register_users(number_of_users, malicious_ratio=0.05):
    """
    Register IoT users for the TSTV experiment.

    Parameters
    ----------
    number_of_users : int
        Number of users to register.
    malicious_ratio : float
        Proportion of users initially marked as malicious
        for the experiment.

    Returns
    -------
    pandas.DataFrame
        Registered user records.
    """

    users = []

    malicious_count = int(number_of_users * malicious_ratio)

    # Select experiment-specific malicious users
    malicious_ids = set(
        random.sample(
            range(1, number_of_users + 1),
            malicious_count
        )
    )

    for user_id in range(1, number_of_users + 1):

        username = f"user_{user_id}"
        password = secrets.token_hex(8)

        user_record = {
            "user_id": user_id,
            "username": username,
            "password_hash": hash_password(password),
            "credential_valid": True,
            "registered": True,
            "malicious": user_id in malicious_ids,
            "authorized": not (user_id in malicious_ids),
            "registration_status": "Registered"
        }

        users.append(user_record)

    return pd.DataFrame(users)


# ------------------------------------------------------------
# Test registration with 100 users
# ------------------------------------------------------------

users_100 = register_users(100)

print("Registration completed.")
print("Total users:", len(users_100))
print("Malicious users:", users_100["malicious"].sum())
print("Normal users:", (~users_100["malicious"]).sum())

display(users_100.head(10))

In [ ]:
# ============================================================
# CELL 3 — TSTV USER CONTROL PANEL (UCP)
# ============================================================

def verify_credentials(user_record, username, password):
    """
    Verify the user's credentials against the registered record.
    """

    if user_record["username"] != username:
        return False

    password_hash = hash_password(password)

    return password_hash == user_record["password_hash"]


def ucp_verification(user_record, username, password):
    """
    User Control Panel (UCP)

    Performs:
    1. Credential verification
    2. Authentication
    3. Authorization
    """

    result = {
        "user_id": user_record["user_id"],
        "credential_verified": False,
        "authenticated": False,
        "authorized": False,
        "status": "Rejected"
    }

    # --------------------------------------------------------
    # Stage 1: Credential verification
    # --------------------------------------------------------

    if not user_record["registered"]:
        result["status"] = "Rejected - User not registered"
        return result

    if not user_record["credential_valid"]:
        result["status"] = "Rejected - Invalid credentials"
        return result

    if not verify_credentials(user_record, username, password):
        result["status"] = "Rejected - Credential mismatch"
        return result

    result["credential_verified"] = True

    # --------------------------------------------------------
    # Stage 2: Authentication
    # --------------------------------------------------------

    result["authenticated"] = True

    # --------------------------------------------------------
    # Stage 3: Authorization
    # --------------------------------------------------------

    if user_record["authorized"] and not user_record["malicious"]:
        result["authorized"] = True
        result["status"] = "UCP Verified"
    else:
        result["status"] = "Rejected - Unauthorized user"

    return result


print("UCP module initialized successfully.")

## 2. Cryptographic environment

In [ ]:
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.hkdf import HKDF

print("Cryptographic environment initialized.")


### 2.1 ECC/ECDH and HKDF session-key establishment

In [ ]:
# ============================================================
# CELL 14 — ECC SESSION-KEY GENERATION
# ============================================================

def generate_ecc_key_pair():
    """
    Generate an ECC private/public key pair.
    """

    private_key = ec.generate_private_key(
        ec.SECP256R1()
    )

    public_key = private_key.public_key()

    return private_key, public_key


def derive_shared_key(private_key, peer_public_key):
    """
    Derive a shared session key using ECC ECDH.
    """

    shared_secret = private_key.exchange(
        ec.ECDH(),
        peer_public_key
    )

    # Derive a 256-bit session key from the shared secret
    session_key = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=None,
        info=b"TSTV-Session-Key"
    ).derive(shared_secret)

    return session_key


# ------------------------------------------------------------
# Simulate two communicating entities
# ------------------------------------------------------------

user_private_key, user_public_key = generate_ecc_key_pair()

cloud_private_key, cloud_public_key = generate_ecc_key_pair()


# User derives session key using cloud public key
user_session_key = derive_shared_key(
    user_private_key,
    cloud_public_key
)


# Cloud derives session key using user public key
cloud_session_key = derive_shared_key(
    cloud_private_key,
    user_public_key
)


# ------------------------------------------------------------
# Verify both sides derived the same session key
# ------------------------------------------------------------

print("User session key  :", user_session_key.hex())
print("Cloud session key :", cloud_session_key.hex())

print(
    "\nSession keys match:",
    user_session_key == cloud_session_key
)

### 2.2 PRESENT-80 implementation

In [ ]:
# ============================================================
# CELL 15 — PRESENT-80 LIGHTWEIGHT BLOCK CIPHER
# ============================================================

# PRESENT uses:
#   - 64-bit block size
#   - 80-bit key in this implementation
#   - 31 rounds

SBOX = [
    0xC, 0x5, 0x6, 0xB,
    0x9, 0x0, 0xA, 0xD,
    0x3, 0xE, 0xF, 0x8,
    0x4, 0x7, 0x1, 0x2
]

PBOX = [
     0, 16, 32, 48,  1, 17, 33, 49,
     2, 18, 34, 50,  3, 19, 35, 51,
     4, 20, 36, 52,  5, 21, 37, 53,
     6, 22, 38, 54,  7, 23, 39, 55,
     8, 24, 40, 56,  9, 25, 41, 57,
    10, 26, 42, 58, 11, 27, 43, 59,
    12, 28, 44, 60, 13, 29, 45, 61,
    14, 30, 46, 62, 15, 31, 47, 63
]


def present_sbox_layer(state):
    """Apply PRESENT S-box to each 4-bit nibble."""

    output = 0

    for i in range(16):
        nibble = (state >> (4 * i)) & 0xF
        output |= SBOX[nibble] << (4 * i)

    return output


def present_pbox_layer(state):
    """Apply PRESENT bit permutation."""

    output = 0

    for i in range(64):
        bit = (state >> i) & 1
        output |= bit << PBOX[i]

    return output


def present_key_update(key, round_counter):
    """
    PRESENT-80 key schedule update.
    """

    # Rotate 80-bit key left by 61 bits
    key = ((key << 61) & ((1 << 80) - 1)) | (key >> 19)

    # Apply S-box to the most significant nibble
    top_nibble = (key >> 76) & 0xF
    key &= ~(0xF << 76)
    key |= SBOX[top_nibble] << 76

    # XOR round counter into bits 19..15
    key ^= round_counter << 15

    return key


def generate_present_round_keys(key):
    """Generate 32 round keys."""

    round_keys = []

    for round_counter in range(1, 33):

        # Most significant 64 bits form round key
        round_key = (key >> 16) & ((1 << 64) - 1)

        round_keys.append(round_key)

        if round_counter <= 31:
            key = present_key_update(
                key,
                round_counter
            )

    return round_keys


def present_encrypt_block(block, key):
    """
    Encrypt one 64-bit block using PRESENT-80.
    """

    round_keys = generate_present_round_keys(key)

    state = block

    for i in range(31):

        state ^= round_keys[i]

        state = present_sbox_layer(state)

        state = present_pbox_layer(state)

    state ^= round_keys[31]

    return state


def present_decrypt_block(block, key):
    """
    Decrypt one 64-bit block using PRESENT-80.
    """

    round_keys = generate_present_round_keys(key)

    state = block

    # Final whitening
    state ^= round_keys[31]

    for i in range(30, -1, -1):

        # Inverse permutation
        inverse_pbox = [0] * 64

        for j, position in enumerate(PBOX):
            inverse_pbox[position] = j

        output = 0

        for j in range(64):
            bit = (state >> j) & 1
            output |= bit << inverse_pbox[j]

        state = output

        # Inverse S-box
        inverse_sbox = [0] * 16

        for j, value in enumerate(SBOX):
            inverse_sbox[value] = j

        output = 0

        for j in range(16):
            nibble = (state >> (4 * j)) & 0xF
            output |= inverse_sbox[nibble] << (4 * j)

        state = output

        state ^= round_keys[i]

    return state


print("PRESENT-80 implementation initialized.")

### 2.3 PRESENT-80 key derivation

In [ ]:
# ============================================================
# CELL 17 — ECC SESSION KEY → PRESENT-80 KEY
# ============================================================

def derive_present_key(session_key):
    """
    Derive an 80-bit PRESENT key from the ECC-derived
    session key.

    The first 10 bytes (80 bits) of the SHA-256-derived
    session material are used as the PRESENT-80 key.
    """

    digest = hashlib.sha256(
        session_key
    ).digest()

    present_key_bytes = digest[:10]

    present_key = int.from_bytes(
        present_key_bytes,
        byteorder="big"
    )

    return present_key


present_key = derive_present_key(
    user_session_key
)

print("Derived PRESENT-80 key:")
print(hex(present_key))

print("\nKey size:", present_key.bit_length(), "bits")

### 2.4 PRESENT-80 data encryption and decryption

In [ ]:
# ============================================================
# CELL 18 — PRESENT ENCRYPTION OF TSTV DATA
# ============================================================

def bytes_to_64bit_blocks(data):
    """
    Convert byte data into 64-bit blocks using zero padding.
    """
    if isinstance(data, str):
        data = data.encode("utf-8")

    # Zero padding to a multiple of 8 bytes
    padding_length = (8 - (len(data) % 8)) % 8
    padded_data = data + b"\x00" * padding_length

    blocks = []

    for i in range(0, len(padded_data), 8):
        block = int.from_bytes(
            padded_data[i:i+8],
            byteorder="big"
        )
        blocks.append(block)

    return blocks


def blocks_to_bytes(blocks):
    """
    Convert 64-bit integer blocks back to bytes.
    """
    output = b""

    for block in blocks:
        output += block.to_bytes(8, byteorder="big")

    return output


def present_encrypt_data(data, key):
    """
    Encrypt arbitrary data using PRESENT-80.
    """

    blocks = bytes_to_64bit_blocks(data)

    encrypted_blocks = [
        present_encrypt_block(block, key)
        for block in blocks
    ]

    return encrypted_blocks


def present_decrypt_data(encrypted_blocks, key):
    """
    Decrypt PRESENT-80 encrypted blocks.
    """

    decrypted_blocks = [
        present_decrypt_block(block, key)
        for block in encrypted_blocks
    ]

    return blocks_to_bytes(decrypted_blocks)


# ------------------------------------------------------------
# Test with TSTV data
# ------------------------------------------------------------

tstv_message = "Temperature=28.5"

encrypted_message = present_encrypt_data(
    tstv_message,
    present_key
)

decrypted_message = present_decrypt_data(
    encrypted_message,
    present_key
)

print("Original data:")
print(tstv_message)

print("\nEncrypted blocks:")
for block in encrypted_message:
    print(hex(block))

print("\nDecrypted data:")
print(decrypted_message.rstrip(b"\x00").decode("utf-8"))

print(
    "\nEncryption/decryption successful:",
    decrypted_message.rstrip(b"\x00").decode("utf-8") == tstv_message
)

### 2.5 Protected TSTV packet and DCP verification

In [ ]:
# ============================================================
# CELL 19 — SECURE TSTV PACKET + HMAC + DCP INTEGRATION
# ============================================================

import json
import time
import hmac
import hashlib



def verify_timestamp(packet):
    """Verify whether the packet is still within its TTL."""
    current_time = time.time()
    elapsed_time = current_time - packet["timestamp"]
    return elapsed_time <= packet["ttl"]


def create_secure_tstv_packet(
    source,
    destination,
    data,
    present_key,
    mac_key,
    ttl=300
):
    """
    Create a secure TSTV data packet.

    Processing:
        1. Encrypt application data using PRESENT-80
        2. Add source and destination
        3. Add timestamp and TTL
        4. Add data flag
        5. Generate keyed HMAC-SHA256
    """

    # --------------------------------------------------------
    # 1. Encrypt application data
    # --------------------------------------------------------

    encrypted_blocks = present_encrypt_data(
        data,
        present_key
    )

    # --------------------------------------------------------
    # 2. Create packet metadata
    # --------------------------------------------------------

    packet = {
        "source": source,
        "destination": destination,
        "encrypted_data": [
            hex(block) for block in encrypted_blocks
        ],
        "timestamp": time.time(),
        "ttl": ttl,
        "data_flag": True
    }

    # --------------------------------------------------------
    # 3. Create canonical representation
    # --------------------------------------------------------

    packet_data = json.dumps(
        packet,
        sort_keys=True,
        separators=(",", ":")
    ).encode("utf-8")

    # --------------------------------------------------------
    # 4. Generate keyed HMAC
    # --------------------------------------------------------

    packet["hmac"] = hmac.new(
        mac_key,
        packet_data,
        hashlib.sha256
    ).hexdigest()

    return packet


def verify_secure_packet_hmac(packet, mac_key):
    """
    Verify HMAC-SHA256 of a received TSTV packet.
    """

    received_hmac = packet.get("hmac")

    if received_hmac is None:
        return False

    packet_without_hmac = {
        key: value
        for key, value in packet.items()
        if key != "hmac"
    }

    packet_data = json.dumps(
        packet_without_hmac,
        sort_keys=True,
        separators=(",", ":")
    ).encode("utf-8")

    expected_hmac = hmac.new(
        mac_key,
        packet_data,
        hashlib.sha256
    ).hexdigest()

    return hmac.compare_digest(
        received_hmac,
        expected_hmac
    )


def decrypt_secure_tstv_packet(packet, present_key):
    """
    Decrypt the encrypted application data
    contained in a secure TSTV packet.
    """

    encrypted_blocks = [
        int(block, 16)
        for block in packet["encrypted_data"]
    ]

    decrypted_data = present_decrypt_data(
        encrypted_blocks,
        present_key
    )

    return decrypted_data.rstrip(
        b"\x00"
    ).decode("utf-8")


def dcp_secure_verification(
    packet,
    expected_source,
    expected_destination,
    mac_key
):
    """
    DCP verification for the secure TSTV packet.

    Checks independently:
        - source
        - destination
        - data flag
        - HMAC
        - TTL
    """

    result = {
        "source_verified": False,
        "destination_verified": False,
        "data_flag_verified": False,
        "hmac_verified": False,
        "ttl_verified": False,
        "status": "DCP Rejected"
    }

    # --------------------------------------------------------
    # Source verification
    # --------------------------------------------------------

    result["source_verified"] = (
        packet.get("source") == expected_source
    )

    # --------------------------------------------------------
    # Destination verification
    # --------------------------------------------------------

    result["destination_verified"] = (
        packet.get("destination") == expected_destination
    )

    # --------------------------------------------------------
    # Data flag verification
    # --------------------------------------------------------

    result["data_flag_verified"] = (
        packet.get("data_flag") is True
    )

    # --------------------------------------------------------
    # HMAC verification
    # --------------------------------------------------------

    result["hmac_verified"] = verify_secure_packet_hmac(
        packet,
        mac_key
    )

    # --------------------------------------------------------
    # TTL verification
    # --------------------------------------------------------

    result["ttl_verified"] = verify_timestamp(packet)

    # --------------------------------------------------------
    # Final DCP decision
    # --------------------------------------------------------

    all_checks_passed = all([
        result["source_verified"],
        result["destination_verified"],
        result["data_flag_verified"],
        result["hmac_verified"],
        result["ttl_verified"]
    ])

    if all_checks_passed:
        result["status"] = "DCP Verified"

    return result


print("Secure TSTV packet and DCP integration initialized.")

### 2.6 Complete TSTV secure transaction

In [ ]:
# ============================================================
# CELL 21 — COMPLETE TSTV SECURE TRANSACTION
# ============================================================

def tstv_secure_transaction(
    user_record,
    username,
    password,
    source,
    destination,
    data,
    expected_source,
    expected_destination,
    session_key,
    ttl=300
):
    """
    Complete TSTV secure transaction.

    Stage 1:
        UCP
        - Credential verification
        - Authentication
        - Authorization

    Cryptographic layer:
        - ECC-derived session key
        - PRESENT-80 data encryption
        - HMAC-SHA256 packet integrity

    Stage 2:
        DCP
        - Source verification
        - Destination verification
        - Data-flag verification
        - HMAC verification
        - TTL verification

    Final:
        Data is decrypted only when UCP and DCP succeed.
    """

    # ========================================================
    # STAGE 1 — UCP
    # ========================================================

    start_time = time.perf_counter()

    ucp_result = ucp_verification(
        user_record,
        username,
        password
    )

    # Stop immediately if UCP fails
    if ucp_result["status"] != "UCP Verified":

        execution_time = time.perf_counter() - start_time

        return {
            "tstv_status": "Rejected",
            "stage": "UCP",
            "ucp_result": ucp_result,
            "dcp_result": None,
            "decrypted_data": None,
            "execution_time": execution_time
        }

    # ========================================================
    # CRYPTOGRAPHIC LAYER
    # ========================================================

    # Derive PRESENT-80 key from ECC session key
    transaction_present_key = derive_present_key(
        session_key
    )

    # Derive HMAC key from ECC session key
    transaction_mac_key = hashlib.sha256(
        b"TSTV-HMAC-" + session_key
    ).digest()

    # Create secure packet
    secure_packet = create_secure_tstv_packet(
        source=source,
        destination=destination,
        data=data,
        present_key=transaction_present_key,
        mac_key=transaction_mac_key,
        ttl=ttl
    )

    # ========================================================
    # STAGE 2 — DCP
    # ========================================================

    dcp_result = dcp_secure_verification(
        packet=secure_packet,
        expected_source=expected_source,
        expected_destination=expected_destination,
        mac_key=transaction_mac_key
    )

    # ========================================================
    # FINAL TSTV DECISION
    # ========================================================

    if dcp_result["status"] == "DCP Verified":

        decrypted_data = decrypt_secure_tstv_packet(
            secure_packet,
            transaction_present_key
        )

        tstv_status = "TSTV Verified"

    else:

        decrypted_data = None
        tstv_status = "Rejected"

    execution_time = time.perf_counter() - start_time

    return {
        "tstv_status": tstv_status,
        "stage": "UCP + Cryptography + DCP",
        "ucp_result": ucp_result,
        "dcp_result": dcp_result,
        "decrypted_data": decrypted_data,
        "execution_time": execution_time,
        "secure_packet": secure_packet
    }


print("Complete TSTV secure transaction initialized.")

## 3. Baseline TSTV performance framework

In [ ]:
# ============================================================
# CELL 24 — TSTV EXPERIMENTAL FRAMEWORK
# ============================================================

EXPERIMENT_USER_SIZES = [100, 200, 300, 400, 500]


def create_experiment_user(user_id):
    """
    Create one legitimate experimental user.
    """

    password = f"TSTV_User_Password_{user_id}"

    return {
        "user_id": user_id,
        "username": f"user_{user_id}",
        "password_hash": hash_password(password),
        "password": password,
        "credential_valid": True,
        "registered": True,
        "malicious": False,
        "authorized": True,
        "registration_status": "Registered"
    }


def run_normal_experiment(number_of_users):
    """
    Run a baseline TSTV experiment using legitimate users
    and valid data packets.
    """

    results = []

    # --------------------------------------------------------
    # 1. Generate legitimate users
    # --------------------------------------------------------

    users = [
        create_experiment_user(user_id)
        for user_id in range(1, number_of_users + 1)
    ]

    # --------------------------------------------------------
    # 2. Generate ECC key pairs
    # --------------------------------------------------------

    experiment_user_private, experiment_user_public = (
        generate_ecc_key_pair()
    )

    experiment_cloud_private, experiment_cloud_public = (
        generate_ecc_key_pair()
    )

    # --------------------------------------------------------
    # 3. Derive shared ECC session key
    # --------------------------------------------------------

    experiment_session_key = derive_shared_key(
        experiment_user_private,
        experiment_cloud_public
    )

    cloud_session_key = derive_shared_key(
        experiment_cloud_private,
        experiment_user_public
    )

    # Verify both sides have the same session key
    assert experiment_session_key == cloud_session_key

    # --------------------------------------------------------
    # 4. Process each user
    # --------------------------------------------------------

    for user in users:

        start = time.perf_counter()

        transaction = tstv_secure_transaction(
            user_record=user,
            username=user["username"],
            password=user["password"],
            source=f"IoT_Device_{user['user_id']}",
            destination="Cloud_Server_01",
            data=f"Temperature={20 + (user['user_id'] % 15)}",
            expected_source=f"IoT_Device_{user['user_id']}",
            expected_destination="Cloud_Server_01",
            session_key=experiment_session_key,
            ttl=300
        )

        elapsed = time.perf_counter() - start

        results.append({
            "user_id": user["user_id"],
            "ucp_status": transaction["ucp_result"]["status"],
            "dcp_status": (
                transaction["dcp_result"]["status"]
                if transaction["dcp_result"] is not None
                else "Not Executed"
            ),
            "tstv_status": transaction["tstv_status"],
            "execution_time": elapsed
        })

    return pd.DataFrame(results)


print("Experimental framework initialized successfully.")

## 4. Attack-load experiment

In [ ]:
# Cell 31C: Corrected configurable balanced attack experiment

def run_balanced_attack_experiment(
    number_of_users,
    attack_ratio=0.20
):

    if not 0 <= attack_ratio <= 1:
        raise ValueError("attack_ratio must be between 0 and 1.")

    # Create legitimate users
    users = [
        create_experiment_user(i)
        for i in range(1, number_of_users + 1)
    ]

    # ECC session context
    user_private, user_public = generate_ecc_key_pair()
    cloud_private, cloud_public = generate_ecc_key_pair()

    session_key = derive_shared_key(
        user_private,
        cloud_public
    )

    # Determine attack workload
    attack_count = int(number_of_users * attack_ratio)
    legitimate_count = number_of_users - attack_count

    attack_types = [
        "Unauthorized User",
        "Invalid Source",
        "Tampered Data",
        "Expired TTL"
    ]

    # Balance attack categories
    base_count = attack_count // len(attack_types)
    remainder = attack_count % len(attack_types)

    attack_list = []

    for i, attack_type in enumerate(attack_types):

        count = base_count + (
            1 if i < remainder else 0
        )

        attack_list.extend(
            [attack_type] * count
        )

    # Create traffic plan
    traffic_plan = (
        ["Legitimate"] * legitimate_count
        + ["Attack"] * attack_count
    )

    # Reproducible shuffling
    random.seed(
        SEED
        + number_of_users
        + int(attack_ratio * 1000)
    )

    random.shuffle(traffic_plan)
    random.shuffle(attack_list)

    attack_index = 0
    results = []

    for user, traffic_type in zip(users, traffic_plan):

        expected_source = f"Device_{user['user_id']}"
        destination = "Cloud_Server"

        data = (
            f"Temperature={20 + (user['user_id'] % 10)}"
        )

        if traffic_type == "Legitimate":

            attack_type = "Legitimate"

            result = tstv_secure_transaction(
                user_record=user,
                username=user["username"],
                password=user["password"],
                source=expected_source,
                expected_source=expected_source,
                destination=destination,
                expected_destination=destination,
                data=data,
                session_key=session_key,
                ttl=300
            )

            detected = None

        else:

            attack_type = attack_list[attack_index]
            attack_index += 1

            if attack_type == "Unauthorized User":

                # Simulate unauthorized access
                user["authorized"] = False

                result = tstv_secure_transaction(
                    user_record=user,
                    username=user["username"],
                    password=user["password"],
                    source=expected_source,
                    expected_source=expected_source,
                    destination=destination,
                    expected_destination=destination,
                    data=data,
                    session_key=session_key,
                    ttl=300
                )

            elif attack_type == "Invalid Source":

                result = tstv_secure_transaction(
                    user_record=user,
                    username=user["username"],
                    password=user["password"],
                    source="Malicious_Device",
                    expected_source=expected_source,
                    destination=destination,
                    expected_destination=destination,
                    data=data,
                    session_key=session_key,
                    ttl=300
                )

            elif attack_type == "Expired TTL":

                result = tstv_secure_transaction(
                    user_record=user,
                    username=user["username"],
                    password=user["password"],
                    source=expected_source,
                    expected_source=expected_source,
                    destination=destination,
                    expected_destination=destination,
                    data=data,
                    session_key=session_key,
                    ttl=-1
                )

            elif attack_type == "Tampered Data":

                # First create a legitimate secure transaction
                legitimate_result = tstv_secure_transaction(
                    user_record=user,
                    username=user["username"],
                    password=user["password"],
                    source=expected_source,
                    expected_source=expected_source,
                    destination=destination,
                    expected_destination=destination,
                    data=data,
                    session_key=session_key,
                    ttl=300
                )

                # Copy the secure packet
                secure_packet = (
                    legitimate_result["secure_packet"].copy()
                )

                secure_packet["encrypted_data"] = (
                    secure_packet["encrypted_data"].copy()
                )

                # Tamper with encrypted payload
                secure_packet["encrypted_data"][0] = 0

                # Derive HMAC key
                hmac_key = hashlib.sha256(
                    b"TSTV-HMAC-" + session_key
                ).digest()

                # Verify tampered packet
                dcp_result = dcp_secure_verification(
                    secure_packet,
                    expected_source,
                    destination,
                    hmac_key
                )

                # IMPORTANT:
                # dcp_secure_verification() returns "status"
                result = {
                    "tstv_status": (
                        "TSTV Verified"
                        if dcp_result["status"] == "DCP Verified"
                        else "Rejected"
                    ),
                    "stage": "DCP",
                    "ucp_result": "UCP Verified",
                    "dcp_result": dcp_result["status"],
                    "decrypted_data": None,
                    "execution_time": (
                        legitimate_result["execution_time"]
                    ),
                    "secure_packet": secure_packet
                }

            detected = (
                result["tstv_status"] == "Rejected"
            )

        results.append({
            "user_id": user["user_id"],
            "traffic_type": traffic_type,
            "attack_type": attack_type,
            "tstv_status": result["tstv_status"],
            "detected": detected,
            "execution_time": result["execution_time"]
        })

    return pd.DataFrame(results)


print(
    "Corrected configurable attack experiment "
    "function is ready."
)

## 5. Payload-size performance experiment

In [ ]:
# Cell 32: Payload-size benchmark — initial validation

PAYLOAD_SIZES = [
    64,
    128,
    256,
    512,
    1024,
    2048,
    4096
]

REPEATS = 10

payload_test_results = []

# Create one test user
benchmark_user = create_experiment_user(9001)

# Create ECC session context
benchmark_private, benchmark_public = generate_ecc_key_pair()
benchmark_cloud_private, benchmark_cloud_public = generate_ecc_key_pair()

benchmark_session_key = derive_shared_key(
    benchmark_private,
    benchmark_cloud_public
)

for payload_size in PAYLOAD_SIZES:

    print(f"Testing payload size: {payload_size} bytes")

    # Generate deterministic payload
    payload = (
        "A" * payload_size
    )

    for repeat in range(REPEATS):

        result = tstv_secure_transaction(
            user_record=benchmark_user,
            username=benchmark_user["username"],
            password=benchmark_user["password"],
            source="Device_9001",
            expected_source="Device_9001",
            destination="Cloud_Server",
            expected_destination="Cloud_Server",
            data=payload,
            session_key=benchmark_session_key,
            ttl=300
        )

        payload_test_results.append({
            "payload_size_bytes": payload_size,
            "repeat": repeat + 1,
            "tstv_status": result["tstv_status"],
            "execution_time": result["execution_time"]
        })

payload_benchmark_df = pd.DataFrame(
    payload_test_results
)

print("\nPayload benchmark completed.")

display(payload_benchmark_df)

### 5.1 Payload-size summary

In [ ]:
# Cell 33: Payload-size performance statistics

payload_summary = []

for payload_size in PAYLOAD_SIZES:

    data = payload_benchmark_df[
        payload_benchmark_df["payload_size_bytes"] == payload_size
    ]

    times = data["execution_time"]

    payload_summary.append({
        "Payload Size (bytes)": payload_size,
        "Trials": len(times),
        "Mean Time (s)": times.mean(),
        "Std Dev (s)": times.std(ddof=1),
        "Min Time (s)": times.min(),
        "Max Time (s)": times.max(),
        "Median Time (s)": times.median()
    })

payload_summary_df = pd.DataFrame(payload_summary)

display(payload_summary_df)

print("\nPayload Performance Summary")
print("---------------------------")

for _, row in payload_summary_df.iterrows():

    print(
        f"{int(row['Payload Size (bytes)'])} bytes: "
        f"Mean={row['Mean Time (s)']:.6f} s, "
        f"SD={row['Std Dev (s)']:.6f} s, "
        f"Min={row['Min Time (s)']:.6f} s, "
        f"Max={row['Max Time (s)']:.6f} s"
    )

### 5.2 Payload-size confidence intervals

In [ ]:
# Cell 34: 95% confidence intervals for payload execution time

from scipy import stats

payload_ci_summary = []

for payload_size in PAYLOAD_SIZES:

    data = payload_benchmark_df[
        payload_benchmark_df["payload_size_bytes"] == payload_size
    ]

    times = data["execution_time"].to_numpy()

    n = len(times)
    mean_time = np.mean(times)
    std_time = np.std(times, ddof=1)

    # 95% confidence interval using the t-distribution
    standard_error = std_time / np.sqrt(n)

    t_critical = stats.t.ppf(
        0.975,
        df=n - 1
    )

    margin_of_error = (
        t_critical * standard_error
    )

    ci_lower = mean_time - margin_of_error
    ci_upper = mean_time + margin_of_error

    payload_ci_summary.append({
        "Payload Size (bytes)": payload_size,
        "Trials": n,
        "Mean Time (s)": mean_time,
        "Std Dev (s)": std_time,
        "95% CI Lower (s)": ci_lower,
        "95% CI Upper (s)": ci_upper
    })

payload_ci_df = pd.DataFrame(
    payload_ci_summary
)

display(payload_ci_df)

print("\n95% Confidence Intervals")
print("------------------------")

for _, row in payload_ci_df.iterrows():

    print(
        f"{int(row['Payload Size (bytes)'])} bytes: "
        f"Mean={row['Mean Time (s)']:.6f} s, "
        f"95% CI="
        f"[{row['95% CI Lower (s)']:.6f}, "
        f"{row['95% CI Upper (s)']:.6f}]"
    )

## 6. End-to-end user-load performance experiment

In [ ]:
# Cell 36: Repeated user-load performance experiment

USER_LOAD_REPEATS = 10

user_load_results = []

for n in EXPERIMENT_USER_SIZES:

    print(f"Running {n}-user experiment...")

    for repeat in range(1, USER_LOAD_REPEATS + 1):

        # Run a fresh legitimate workload
        result = run_normal_experiment(n)

        # Mean transaction execution time for this trial
        mean_time = result["execution_time"].mean()

        # Standard deviation within this trial
        std_time = result["execution_time"].std(ddof=1)

        # Verification success
        verified_count = (
            result["tstv_status"] == "TSTV Verified"
        ).sum()

        total_count = len(result)

        verification_rate = (
            verified_count / total_count
            if total_count > 0 else 0
        )

        user_load_results.append({
            "Users": n,
            "Repeat": repeat,
            "Transactions": total_count,
            "Mean Execution Time (s)": mean_time,
            "Within-Trial SD (s)": std_time,
            "Verified Transactions": verified_count,
            "Verification Rate": verification_rate
        })

user_load_df = pd.DataFrame(user_load_results)

print("\nUser-load experiment completed.")

display(user_load_df)

### 6.1 User-load confidence intervals

In [ ]:
# Cell 37: User-load performance statistics with 95% CI

from scipy import stats

user_load_summary = []

for n in EXPERIMENT_USER_SIZES:

    data = user_load_df[
        user_load_df["Users"] == n
    ]

    # These are the 10 independent trial means
    times = data["Mean Execution Time (s)"].to_numpy()

    trials = len(times)
    mean_time = np.mean(times)
    std_time = np.std(times, ddof=1)

    standard_error = std_time / np.sqrt(trials)

    t_critical = stats.t.ppf(
        0.975,
        df=trials - 1
    )

    margin_of_error = (
        t_critical * standard_error
    )

    ci_lower = mean_time - margin_of_error
    ci_upper = mean_time + margin_of_error

    verification_rate = data[
        "Verification Rate"
    ].mean()

    user_load_summary.append({
        "Users": n,
        "Trials": trials,
        "Mean Execution Time (s)": mean_time,
        "Between-Trial SD (s)": std_time,
        "95% CI Lower (s)": ci_lower,
        "95% CI Upper (s)": ci_upper,
        "Mean Verification Rate": verification_rate
    })

user_load_summary_df = pd.DataFrame(
    user_load_summary
)

display(user_load_summary_df)

print("\nUser-Load Performance Summary")
print("-----------------------------")

for _, row in user_load_summary_df.iterrows():

    print(
        f"{int(row['Users'])} users: "
        f"Mean={row['Mean Execution Time (s)']:.6f} s, "
        f"SD={row['Between-Trial SD (s)']:.6f} s, "
        f"95% CI="
        f"[{row['95% CI Lower (s)']:.6f}, "
        f"{row['95% CI Upper (s)']:.6f}], "
        f"Verification="
        f"{row['Mean Verification Rate']*100:.2f}%"
    )

## 7. PRESENT-80 implementation validation

In [ ]:
# Cell 39: Validate PRESENT-80 implementation against a standard test vector

TEST_KEY = 0x00000000000000000000
TEST_PLAINTEXT = 0x0000000000000000
EXPECTED_CIPHERTEXT = 0x5579C1387B228445

# Encrypt
test_ciphertext = present_encrypt_block(
    TEST_PLAINTEXT,
    TEST_KEY
)

# Decrypt
test_decrypted = present_decrypt_block(
    test_ciphertext,
    TEST_KEY
)

print("PRESENT-80 Test Vector")
print("----------------------")

print(f"Key:               {TEST_KEY:020X}")
print(f"Plaintext:         {TEST_PLAINTEXT:016X}")
print(f"Expected cipher:   {EXPECTED_CIPHERTEXT:016X}")
print(f"Computed cipher:   {test_ciphertext:016X}")
print(f"Decrypted plaintext: {test_decrypted:016X}")

encryption_match = (
    test_ciphertext == EXPECTED_CIPHERTEXT
)

decryption_match = (
    test_decrypted == TEST_PLAINTEXT
)

print("\nValidation")
print("----------")
print(f"Encryption test passed: {encryption_match}")
print(f"Decryption test passed: {decryption_match}")

if encryption_match and decryption_match:
    print("\nPRESENT-80 implementation validation PASSED.")
else:
    print("\nPRESENT-80 implementation validation FAILED.")

## 8. Registration performance

In [ ]:
# Cell 41: Registration-stage performance experiment

REGISTRATION_SIZES = [100, 200, 300, 400, 500]
REGISTRATION_REPEATS = 10

registration_results = []

for n in REGISTRATION_SIZES:

    print(f"Running registration experiment: {n} users...")

    for repeat in range(1, REGISTRATION_REPEATS + 1):

        start_time = time.perf_counter()

        registered_users = [
            create_experiment_user(i)
            for i in range(1, n + 1)
        ]

        end_time = time.perf_counter()

        execution_time = end_time - start_time

        successful = sum(
            user["registered"]
            for user in registered_users
        )

        registration_rate = (
            successful / n
            if n > 0 else 0
        )

        registration_results.append({
            "Users": n,
            "Repeat": repeat,
            "Registered Users": successful,
            "Registration Rate": registration_rate,
            "Total Execution Time (s)": execution_time,
            "Mean Time per User (s)": execution_time / n
        })

registration_df = pd.DataFrame(
    registration_results
)

print("\nRegistration experiment completed.")

display(registration_df)

### 8.1 Registration confidence intervals

In [ ]:
# Cell 42: Registration performance statistics with 95% CI

from scipy import stats

registration_summary = []

for n in REGISTRATION_SIZES:

    data = registration_df[
        registration_df["Users"] == n
    ]

    # Independent repeated-trial measurements
    times = data["Total Execution Time (s)"].to_numpy()

    trials = len(times)

    mean_total = np.mean(times)
    std_total = np.std(times, ddof=1)

    standard_error = std_total / np.sqrt(trials)

    t_critical = stats.t.ppf(
        0.975,
        df=trials - 1
    )

    margin_of_error = (
        t_critical * standard_error
    )

    ci_lower = mean_total - margin_of_error
    ci_upper = mean_total + margin_of_error

    # Per-user timing
    mean_per_user = (
        data["Mean Time per User (s)"].mean()
    )

    # Registration success
    registration_rate = (
        data["Registration Rate"].mean()
    )

    registration_summary.append({
        "Users": n,
        "Trials": trials,
        "Mean Total Time (s)": mean_total,
        "Between-Trial SD (s)": std_total,
        "95% CI Lower (s)": ci_lower,
        "95% CI Upper (s)": ci_upper,
        "Mean Time per User (s)": mean_per_user,
        "Registration Rate": registration_rate
    })

registration_summary_df = pd.DataFrame(
    registration_summary
)

display(registration_summary_df)

print("\nRegistration Performance Summary")
print("--------------------------------")

for _, row in registration_summary_df.iterrows():

    print(
        f"{int(row['Users'])} users: "
        f"Mean Total="
        f"{row['Mean Total Time (s)']:.6f} s, "
        f"SD="
        f"{row['Between-Trial SD (s)']:.6f} s, "
        f"95% CI="
        f"[{row['95% CI Lower (s)']:.6f}, "
        f"{row['95% CI Upper (s)']:.6f}], "
        f"Mean/User="
        f"{row['Mean Time per User (s)']:.9f} s, "
        f"Registration="
        f"{row['Registration Rate']*100:.2f}%"
    )

## 9. UCP authentication performance

In [ ]:
# Cell 43: UCP login/authentication performance experiment

UCP_SIZES = [100, 200, 300, 400, 500]
UCP_REPEATS = 10

ucp_results = []

for n in UCP_SIZES:

    print(f"Running UCP experiment: {n} users...")

    for repeat in range(1, UCP_REPEATS + 1):

        # Create registered users
        users = [
            create_experiment_user(i)
            for i in range(1, n + 1)
        ]

        start_time = time.perf_counter()

        verification_results = []

        for user in users:

            result = ucp_verification(
                user_record=user,
                username=user["username"],
                password=user["password"]
            )

            verification_results.append(result)

        end_time = time.perf_counter()

        execution_time = end_time - start_time

        # Count successful UCP verification
        successful = sum(
            result["status"] == "UCP Verified"
            for result in verification_results
        )

        verification_rate = (
            successful / n
            if n > 0 else 0
        )

        ucp_results.append({
            "Users": n,
            "Repeat": repeat,
            "Authentication Attempts": n,
            "UCP Verified": successful,
            "UCP Verification Rate": verification_rate,
            "Total Execution Time (s)": execution_time,
            "Mean Time per User (s)": execution_time / n
        })

ucp_df = pd.DataFrame(
    ucp_results
)

print("\nUCP experiment completed.")

display(ucp_df)

### 9.1 UCP confidence intervals

In [ ]:
from scipy import stats

ucp_summary = []

for n in UCP_SIZES:

    data = ucp_df[
        ucp_df["Users"] == n
    ]

    times = data[
        "Total Execution Time (s)"
    ].to_numpy()

    trials = len(times)

    mean_total = np.mean(times)
    std_total = np.std(times, ddof=1)

    standard_error = (
        std_total / np.sqrt(trials)
    )

    t_critical = stats.t.ppf(
        0.975,
        df=trials - 1
    )

    margin = (
        t_critical * standard_error
    )

    ci_lower = mean_total - margin
    ci_upper = mean_total + margin

    mean_per_user = data[
        "Mean Time per User (s)"
    ].mean()

    verification_rate = data[
        "UCP Verification Rate"
    ].mean()

    ucp_summary.append({
        "Users": n,
        "Trials": trials,
        "Mean Total Time (s)": mean_total,
        "Between-Trial SD (s)": std_total,
        "95% CI Lower (s)": ci_lower,
        "95% CI Upper (s)": ci_upper,
        "Mean Time per User (s)": mean_per_user,
        "UCP Verification Rate": verification_rate
    })

ucp_summary_df = pd.DataFrame(
    ucp_summary
)

display(
    ucp_summary_df.round(6)
)

print("\nUCP Authentication Performance Summary")

for _, row in ucp_summary_df.iterrows():
    print(
        f"{int(row['Users'])} users: "
        f"Mean Total={row['Mean Total Time (s)']:.6f} s, "
        f"SD={row['Between-Trial SD (s)']:.6f} s, "
        f"95% CI=["
        f"{row['95% CI Lower (s)']:.6f}, "
        f"{row['95% CI Upper (s)']:.6f}], "
        f"Mean/User={row['Mean Time per User (s)']:.9f} s, "
        f"UCP Verification="
        f"{row['UCP Verification Rate']*100:.2f}%"
    )

## 10. DCP verification performance

In [ ]:


# Cell 45: DCP / cloud-side secure transaction verification experiment

DCP_SIZES = [100, 200, 300, 400, 500]
DCP_REPEATS = 10

dcp_results = []

# Cloud ECC key pair
cloud_private_key, cloud_public_key = generate_ecc_key_pair()

for n in DCP_SIZES:

    print(f"Running DCP experiment: {n} users...")

    for repeat in range(1, DCP_REPEATS + 1):

        # Create registered legitimate users
        users = [
            create_experiment_user(i)
            for i in range(1, n + 1)
        ]

        # ---------------------------------------------------------
        # Prepare secure packets BEFORE timing DCP verification.
        # Each user has an independent ECC-derived session key.
        # ---------------------------------------------------------
        prepared_packets = []

        for user in users:

            # Independent user ECC key pair
            user_private_key, user_public_key = generate_ecc_key_pair()

            # Independent session key
            session_key = derive_shared_key(
                user_private_key,
                cloud_public_key
            )

            # Derive PRESENT key
            present_key = derive_present_key(
                session_key
            )

            # Derive HMAC key
            mac_key = hashlib.sha256(
                b"TSTV-HMAC-" + session_key
            ).digest()

            # Create secure packet using the existing function
            packet = create_secure_tstv_packet(
                user["username"],
                "cloud",
                f"Secure IoT data from user {user['user_id']}",
                present_key,
                mac_key,
                ttl=300
            )

            prepared_packets.append({
                "user": user,
                "packet": packet,
                "mac_key": mac_key
            })

        # ---------------------------------------------------------
        # Time ONLY DCP verification.
        # ---------------------------------------------------------
        start_time = time.perf_counter()

        verification_results = []

        for item in prepared_packets:

            result = dcp_secure_verification(
                item["packet"],
                item["user"]["username"],
                "cloud",
                item["mac_key"]
            )

            verification_results.append(result)

        end_time = time.perf_counter()

        execution_time = end_time - start_time

        # Count successful DCP verification
        successful = sum(
            result["status"] == "DCP Verified"
            for result in verification_results
        )

        verification_rate = successful / n

        dcp_results.append({
            "Users": n,
            "Repeat": repeat,
            "Transactions": n,
            "DCP Verified": successful,
            "DCP Verification Rate": verification_rate,
            "Total Execution Time (s)": execution_time,
            "Mean Time per Transaction (s)": execution_time / n
        })

dcp_df = pd.DataFrame(dcp_results)

print("\nDCP experiment completed.")

display(dcp_df)

### 10.1 DCP confidence intervals

In [ ]:
from scipy import stats

dcp_summary = []

for n in DCP_SIZES:

    data = dcp_df[
        dcp_df["Users"] == n
    ]

    times = data[
        "Total Execution Time (s)"
    ].to_numpy()

    trials = len(times)

    mean_total = np.mean(times)
    std_total = np.std(times, ddof=1)

    standard_error = (
        std_total / np.sqrt(trials)
    )

    t_critical = stats.t.ppf(
        0.975,
        df=trials - 1
    )

    margin = (
        t_critical * standard_error
    )

    ci_lower = mean_total - margin
    ci_upper = mean_total + margin

    mean_per_transaction = data[
        "Mean Time per Transaction (s)"
    ].mean()

    verification_rate = data[
        "DCP Verification Rate"
    ].mean()

    dcp_summary.append({
        "Users": n,
        "Trials": trials,
        "Mean Total Time (s)": mean_total,
        "Between-Trial SD (s)": std_total,
        "95% CI Lower (s)": ci_lower,
        "95% CI Upper (s)": ci_upper,
        "Mean Time per Transaction (s)": mean_per_transaction,
        "DCP Verification Rate": verification_rate
    })

dcp_summary_df = pd.DataFrame(
    dcp_summary
)

display(
    dcp_summary_df.round(6)
)

print("\nDCP Verification Performance Summary")

for _, row in dcp_summary_df.iterrows():
    print(
        f"{int(row['Users'])} users: "
        f"Mean Total={row['Mean Total Time (s)']:.6f} s, "
        f"SD={row['Between-Trial SD (s)']:.6f} s, "
        f"95% CI=["
        f"{row['95% CI Lower (s)']:.6f}, "
        f"{row['95% CI Upper (s)']:.6f}], "
        f"Mean/Transaction="
        f"{row['Mean Time per Transaction (s)']:.9f} s, "
        f"DCP Verification="
        f"{row['DCP Verification Rate']*100:.2f}%"
    )

## 11. End-to-end TSTV performance

In [ ]:
# Cell 48: End-to-end TSTV performance experiment
#
# Measures the complete TSTV transaction:
# UCP -> cryptographic processing -> DCP -> decryption
#
# Each user receives an independent ECC-derived session key.

TSTV_SIZES = [100, 200, 300, 400, 500]
TSTV_REPEATS = 10

tstv_results = []

# Cloud ECC key pair
cloud_private_key, cloud_public_key = generate_ecc_key_pair()

for n in TSTV_SIZES:

    print(f"Running end-to-end TSTV experiment: {n} users...")

    for repeat in range(1, TSTV_REPEATS + 1):

        # ---------------------------------------------------------
        # Create registered legitimate users
        # ---------------------------------------------------------
        users = [
            create_experiment_user(i)
            for i in range(1, n + 1)
        ]

        # ---------------------------------------------------------
        # Prepare an independent ECC session key for every user.
        # Key establishment is performed BEFORE the timed
        # transaction loop so that the benchmark focuses on the
        # implemented TSTV transaction workflow.
        # ---------------------------------------------------------
        user_sessions = []

        for user in users:

            user_private_key, user_public_key = generate_ecc_key_pair()

            session_key = derive_shared_key(
                user_private_key,
                cloud_public_key
            )

            user_sessions.append({
                "user": user,
                "session_key": session_key
            })

        # ---------------------------------------------------------
        # End-to-end TSTV timing
        # ---------------------------------------------------------
        start_time = time.perf_counter()

        verification_results = []

        for item in user_sessions:

            user = item["user"]
            session_key = item["session_key"]

            result = tstv_secure_transaction(
                user_record=user,
                username=user["username"],
                password=user["password"],
                source=user["username"],
                destination="cloud",
                data=f"Secure IoT data from user {user['user_id']}",
                expected_source=user["username"],
                expected_destination="cloud",
                session_key=session_key,
                ttl=300
            )

            verification_results.append(result)

        end_time = time.perf_counter()

        execution_time = end_time - start_time

        # ---------------------------------------------------------
        # Count successful complete TSTV transactions
        # ---------------------------------------------------------
        successful = sum(
            result["tstv_status"] == "TSTV Verified"
            for result in verification_results
        )

        verification_rate = successful / n

        tstv_results.append({
            "Users": n,
            "Repeat": repeat,
            "Transactions": n,
            "TSTV Verified": successful,
            "TSTV Verification Rate": verification_rate,
            "Total Execution Time (s)": execution_time,
            "Mean Time per Transaction (s)": execution_time / n
        })

tstv_df = pd.DataFrame(tstv_results)

print("\nEnd-to-end TSTV experiment completed.")

display(tstv_df)

### 11.1 End-to-end TSTV confidence intervals

In [ ]:
# Cell 49: End-to-end TSTV statistical summary

from scipy import stats

tstv_summary = []

for n in TSTV_SIZES:

    data = tstv_df[
        tstv_df["Users"] == n
    ]

    times = data[
        "Total Execution Time (s)"
    ].to_numpy()

    trials = len(times)

    mean_total = np.mean(times)
    std_total = np.std(times, ddof=1)

    standard_error = (
        std_total / np.sqrt(trials)
    )

    t_critical = stats.t.ppf(
        0.975,
        df=trials - 1
    )

    margin = (
        t_critical * standard_error
    )

    ci_lower = mean_total - margin
    ci_upper = mean_total + margin

    mean_per_transaction = data[
        "Mean Time per Transaction (s)"
    ].mean()

    verification_rate = data[
        "TSTV Verification Rate"
    ].mean()

    tstv_summary.append({
        "Users": n,
        "Trials": trials,
        "Mean Total Time (s)": mean_total,
        "Between-Trial SD (s)": std_total,
        "95% CI Lower (s)": ci_lower,
        "95% CI Upper (s)": ci_upper,
        "Mean Time per Transaction (s)": mean_per_transaction,
        "TSTV Verification Rate": verification_rate
    })

tstv_summary_df = pd.DataFrame(
    tstv_summary
)

display(
    tstv_summary_df.round(6)
)

print("\nEnd-to-End TSTV Performance Summary")

for _, row in tstv_summary_df.iterrows():

    print(
        f"{int(row['Users'])} users: "
        f"Mean Total={row['Mean Total Time (s)']:.6f} s, "
        f"SD={row['Between-Trial SD (s)']:.6f} s, "
        f"95% CI=["
        f"{row['95% CI Lower (s)']:.6f}, "
        f"{row['95% CI Upper (s)']:.6f}], "
        f"Mean/Transaction="
        f"{row['Mean Time per Transaction (s)']:.9f} s, "
        f"TSTV Verification="
        f"{row['TSTV Verification Rate']*100:.2f}%"
    )

## 12. Systematic security validation

In [ ]:
# Cell 51: Systematic security-validation experiment
#
# Tests the security checks implemented in the actual TSTV code:
# 1. Invalid credentials
# 2. Unauthorized user
# 3. Invalid source
# 4. Invalid destination
# 5. Tampered encrypted data
# 6. Forged/modified HMAC
# 7. Invalid data flag
# 8. Expired TTL

import copy
import time

security_results = []

# ---------------------------------------------------------
# Prepare a legitimate test user and cryptographic context
# ---------------------------------------------------------

test_user = create_experiment_user(1)

test_private_key, test_public_key = generate_ecc_key_pair()

test_session_key = derive_shared_key(
    test_private_key,
    cloud_public_key
)

test_present_key = derive_present_key(
    test_session_key
)

test_mac_key = hashlib.sha256(
    b"TSTV-HMAC-" + test_session_key
).digest()

# ---------------------------------------------------------
# Create a legitimate packet
# ---------------------------------------------------------

legitimate_packet = create_secure_tstv_packet(
    test_user["username"],
    "cloud",
    "Secure IoT data",
    test_present_key,
    test_mac_key,
    ttl=300
)

# =========================================================
# Attack 1 — Invalid credentials
# =========================================================

result = ucp_verification(
    test_user,
    test_user["username"],
    "WRONG_PASSWORD"
)

security_results.append({
    "Attack": "Invalid credentials",
    "Expected Detection Stage": "UCP",
    "Detection Check": "Credential verification",
    "Observed Status": result["status"],
    "Detected": result["status"] != "UCP Verified"
})

# =========================================================
# Attack 2 — Unauthorized user
# =========================================================

unauthorized_user = copy.deepcopy(test_user)
unauthorized_user["authorized"] = False

result = ucp_verification(
    unauthorized_user,
    unauthorized_user["username"],
    unauthorized_user["password"]
)

security_results.append({
    "Attack": "Unauthorized user",
    "Expected Detection Stage": "UCP",
    "Detection Check": "Authorization",
    "Observed Status": result["status"],
    "Detected": result["status"] != "UCP Verified"
})

# =========================================================
# Attack 3 — Invalid source
# =========================================================

packet = copy.deepcopy(legitimate_packet)

result = dcp_secure_verification(
    packet,
    "attacker_source",
    "cloud",
    test_mac_key
)

security_results.append({
    "Attack": "Invalid source",
    "Expected Detection Stage": "DCP",
    "Detection Check": "Source verification",
    "Observed Status": result["status"],
    "Detected": result["status"] != "DCP Verified"
})

# =========================================================
# Attack 4 — Invalid destination
# =========================================================

packet = copy.deepcopy(legitimate_packet)

result = dcp_secure_verification(
    packet,
    test_user["username"],
    "attacker_destination",
    test_mac_key
)

security_results.append({
    "Attack": "Invalid destination",
    "Expected Detection Stage": "DCP",
    "Detection Check": "Destination verification",
    "Observed Status": result["status"],
    "Detected": result["status"] != "DCP Verified"
})

# =========================================================
# Attack 5 — Tampered encrypted data
# =========================================================

packet = copy.deepcopy(legitimate_packet)

# Modify the first encrypted block
original_block = packet["encrypted_data"][0]

if original_block != "0x0000000000000000":
    packet["encrypted_data"][0] = "0x0000000000000000"
else:
    packet["encrypted_data"][0] = "0xFFFFFFFFFFFFFFFF"

result = dcp_secure_verification(
    packet,
    test_user["username"],
    "cloud",
    test_mac_key
)

security_results.append({
    "Attack": "Tampered encrypted data",
    "Expected Detection Stage": "DCP",
    "Detection Check": "HMAC verification",
    "Observed Status": result["status"],
    "Detected": result["status"] != "DCP Verified"
})

# =========================================================
# Attack 6 — Forged / modified HMAC
# =========================================================

packet = copy.deepcopy(legitimate_packet)

# Change the HMAC without changing the encrypted data
packet["hmac"] = (
    "0" * len(packet["hmac"])
)

result = dcp_secure_verification(
    packet,
    test_user["username"],
    "cloud",
    test_mac_key
)

security_results.append({
    "Attack": "Forged/modified HMAC",
    "Expected Detection Stage": "DCP",
    "Detection Check": "HMAC verification",
    "Observed Status": result["status"],
    "Detected": result["status"] != "DCP Verified"
})

# =========================================================
# Attack 7 — Invalid data flag
# =========================================================

packet = copy.deepcopy(legitimate_packet)

packet["data_flag"] = False

result = dcp_secure_verification(
    packet,
    test_user["username"],
    "cloud",
    test_mac_key
)

security_results.append({
    "Attack": "Invalid data flag",
    "Expected Detection Stage": "DCP",
    "Detection Check": "Data-flag verification",
    "Observed Status": result["status"],
    "Detected": result["status"] != "DCP Verified"
})

# =========================================================
# Attack 8 — Expired TTL
# =========================================================

packet = copy.deepcopy(legitimate_packet)

# Make timestamp old enough to exceed the 300-second TTL
packet["timestamp"] = time.time() - 1000

result = dcp_secure_verification(
    packet,
    test_user["username"],
    "cloud",
    test_mac_key
)

security_results.append({
    "Attack": "Expired TTL",
    "Expected Detection Stage": "DCP",
    "Detection Check": "TTL/timestamp verification",
    "Observed Status": result["status"],
    "Detected": result["status"] != "DCP Verified"
})

# =========================================================
# Create final security-validation table
# =========================================================

security_validation_df = pd.DataFrame(
    security_results
)

display(security_validation_df)

# ---------------------------------------------------------
# Overall detection result
# ---------------------------------------------------------

total_attacks = len(security_validation_df)
detected_attacks = security_validation_df["Detected"].sum()
detection_rate = detected_attacks / total_attacks

print("\nSecurity Validation Summary")
print(f"Attack scenarios tested: {total_attacks}")
print(f"Attack scenarios detected: {detected_attacks}")
print(f"Detection rate: {detection_rate * 100:.2f}%")

## 13. ECC/ECDH + HKDF session-key establishment performance

In [ ]:
# Cell 52: ECC/ECDH session-key establishment overhead experiment
#
# Measures the time required to establish an independent ECC-derived
# session key for each user.
#
# This is measured separately from the TSTV transaction itself.

ECC_SIZES = [100, 200, 300, 400, 500]
ECC_REPEATS = 10

ecc_results = []

for n in ECC_SIZES:

    print(f"Running ECC/ECDH experiment: {n} users...")

    for repeat in range(1, ECC_REPEATS + 1):

        # ---------------------------------------------------------
        # Generate one cloud key pair before timing.
        # ---------------------------------------------------------
        cloud_private_key, cloud_public_key = generate_ecc_key_pair()

        # Generate independent user key pairs before timing.
        # Key-pair generation is deliberately excluded so that
        # this experiment measures ECDH session-key derivation.
        user_key_pairs = []

        for _ in range(n):

            user_private_key, user_public_key = generate_ecc_key_pair()

            user_key_pairs.append({
                "private_key": user_private_key,
                "public_key": user_public_key
            })

        # ---------------------------------------------------------
        # Time only ECDH + HKDF session-key derivation.
        # ---------------------------------------------------------
        start_time = time.perf_counter()

        session_keys = []

        for item in user_key_pairs:

            session_key = derive_shared_key(
                item["private_key"],
                cloud_public_key
            )

            session_keys.append(session_key)

        end_time = time.perf_counter()

        execution_time = end_time - start_time

        # ---------------------------------------------------------
        # Validate that every session key was generated correctly.
        # ---------------------------------------------------------
        valid_keys = sum(
            isinstance(key, bytes) and len(key) == 32
            for key in session_keys
        )

        key_generation_rate = valid_keys / n

        ecc_results.append({
            "Users": n,
            "Repeat": repeat,
            "Session Keys Generated": valid_keys,
            "Session-Key Generation Rate": key_generation_rate,
            "Total ECDH/HKDF Time (s)": execution_time,
            "Mean Time per Session Key (s)": execution_time / n
        })

ecc_df = pd.DataFrame(ecc_results)

print("\nECC/ECDH session-key experiment completed.")

display(ecc_df)

### 13.1 ECC/ECDH confidence intervals

In [ ]:
# Cell 53: ECC/ECDH session-key establishment statistical summary

from scipy import stats

ecc_summary = []

for n in ECC_SIZES:

    data = ecc_df[
        ecc_df["Users"] == n
    ]

    times = data[
        "Total ECDH/HKDF Time (s)"
    ].to_numpy()

    trials = len(times)

    mean_total = np.mean(times)
    std_total = np.std(times, ddof=1)

    standard_error = (
        std_total / np.sqrt(trials)
    )

    t_critical = stats.t.ppf(
        0.975,
        df=trials - 1
    )

    margin = (
        t_critical * standard_error
    )

    ci_lower = mean_total - margin
    ci_upper = mean_total + margin

    mean_per_session = data[
        "Mean Time per Session Key (s)"
    ].mean()

    generation_rate = data[
        "Session-Key Generation Rate"
    ].mean()

    ecc_summary.append({
        "Users": n,
        "Trials": trials,
        "Mean Total ECDH/HKDF Time (s)": mean_total,
        "Between-Trial SD (s)": std_total,
        "95% CI Lower (s)": ci_lower,
        "95% CI Upper (s)": ci_upper,
        "Mean Time per Session Key (s)": mean_per_session,
        "Session-Key Generation Rate": generation_rate
    })

ecc_summary_df = pd.DataFrame(
    ecc_summary
)

display(
    ecc_summary_df.round(6)
)

print("\nECC/ECDH Session-Key Establishment Summary")

for _, row in ecc_summary_df.iterrows():

    print(
        f"{int(row['Users'])} users: "
        f"Mean Total={row['Mean Total ECDH/HKDF Time (s)']:.6f} s, "
        f"SD={row['Between-Trial SD (s)']:.6f} s, "
        f"95% CI=["
        f"{row['95% CI Lower (s)']:.6f}, "
        f"{row['95% CI Upper (s)']:.6f}], "
        f"Mean/Session="
        f"{row['Mean Time per Session Key (s)']:.9f} s, "
        f"Generation="
        f"{row['Session-Key Generation Rate']*100:.2f}%"
    )

## 14. Consolidated performance results

In [ ]:
# Cell 54: Consolidate all major performance results
#
# Combines the completed summary dataframes into one
# publication-ready master performance table.

# ---------------------------------------------------------
# Prepare registration summary
# ---------------------------------------------------------

registration_master = registration_summary_df[
    [
        "Users",
        "Trials",
        "Mean Total Time (s)",
        "Between-Trial SD (s)",
        "95% CI Lower (s)",
        "95% CI Upper (s)",
        "Mean Time per User (s)",
        "Registration Rate"
    ]
].copy()

registration_master["Experiment"] = "Registration"
registration_master["Verification Rate"] = (
    registration_master["Registration Rate"]
)

registration_master["Mean Time per Transaction (s)"] = (
    registration_master["Mean Time per User (s)"]
)

# ---------------------------------------------------------
# Prepare UCP summary
# ---------------------------------------------------------

ucp_master = ucp_summary_df[
    [
        "Users",
        "Trials",
        "Mean Total Time (s)",
        "Between-Trial SD (s)",
        "95% CI Lower (s)",
        "95% CI Upper (s)",
        "Mean Time per User (s)",
        "UCP Verification Rate"
    ]
].copy()

ucp_master["Experiment"] = "UCP Authentication"
ucp_master["Verification Rate"] = (
    ucp_master["UCP Verification Rate"]
)

ucp_master["Mean Time per Transaction (s)"] = (
    ucp_master["Mean Time per User (s)"]
)

# ---------------------------------------------------------
# Prepare DCP summary
# ---------------------------------------------------------

dcp_master = dcp_summary_df[
    [
        "Users",
        "Trials",
        "Mean Total Time (s)",
        "Between-Trial SD (s)",
        "95% CI Lower (s)",
        "95% CI Upper (s)",
        "Mean Time per Transaction (s)",
        "DCP Verification Rate"
    ]
].copy()

dcp_master["Experiment"] = "DCP Verification"
dcp_master["Verification Rate"] = (
    dcp_master["DCP Verification Rate"]
)

dcp_master["Mean Time per User (s)"] = (
    dcp_master["Mean Time per Transaction (s)"]
)

# ---------------------------------------------------------
# Prepare end-to-end TSTV summary
# ---------------------------------------------------------

tstv_master = tstv_summary_df[
    [
        "Users",
        "Trials",
        "Mean Total Time (s)",
        "Between-Trial SD (s)",
        "95% CI Lower (s)",
        "95% CI Upper (s)",
        "Mean Time per Transaction (s)",
        "TSTV Verification Rate"
    ]
].copy()

tstv_master["Experiment"] = "End-to-End TSTV"
tstv_master["Verification Rate"] = (
    tstv_master["TSTV Verification Rate"]
)

tstv_master["Mean Time per User (s)"] = (
    tstv_master["Mean Time per Transaction (s)"]
)

# ---------------------------------------------------------
# Prepare ECC/ECDH summary
# ---------------------------------------------------------

ecc_master = ecc_summary_df[
    [
        "Users",
        "Trials",
        "Mean Total ECDH/HKDF Time (s)",
        "Between-Trial SD (s)",
        "95% CI Lower (s)",
        "95% CI Upper (s)",
        "Mean Time per Session Key (s)",
        "Session-Key Generation Rate"
    ]
].copy()

ecc_master["Experiment"] = "ECC/ECDH + HKDF"

ecc_master["Mean Total Time (s)"] = (
    ecc_master["Mean Total ECDH/HKDF Time (s)"]
)

ecc_master["Verification Rate"] = (
    ecc_master["Session-Key Generation Rate"]
)

ecc_master["Mean Time per User (s)"] = (
    ecc_master["Mean Time per Session Key (s)"]
)

# ---------------------------------------------------------
# Align columns
# ---------------------------------------------------------

common_columns = [
    "Experiment",
    "Users",
    "Trials",
    "Mean Total Time (s)",
    "Between-Trial SD (s)",
    "95% CI Lower (s)",
    "95% CI Upper (s)",
    "Mean Time per User (s)",
    "Mean Time per Transaction (s)",
    "Verification Rate"
]

registration_master = registration_master.reindex(
    columns=common_columns
)

ucp_master = ucp_master.reindex(
    columns=common_columns
)

dcp_master = dcp_master.reindex(
    columns=common_columns
)

tstv_master = tstv_master.reindex(
    columns=common_columns
)

ecc_master = ecc_master.reindex(
    columns=common_columns
)

# ---------------------------------------------------------
# Combine all experiments
# ---------------------------------------------------------

master_performance_df = pd.concat(
    [
        registration_master,
        ucp_master,
        dcp_master,
        tstv_master,
        ecc_master
    ],
    ignore_index=True
)

# ---------------------------------------------------------
# Display rounded publication-oriented table
# ---------------------------------------------------------

display(
    master_performance_df.round(6)
)

print("\nMaster performance dataset created.")

print(
    f"Experiments included: "
    f"{master_performance_df['Experiment'].nunique()}"
)

print(
    f"Total summary rows: "
    f"{len(master_performance_df)}"
)

### 14.1 Publication-ready performance table

In [ ]:
# Cell 55: Publication-ready principal performance table
#
# This table summarizes the main measured TSTV components.
# Confidence intervals remain available in the underlying
# master dataset and can be reported in the text or supplementary
# material if needed.

publication_rows = []

for experiment in [
    "Registration",
    "UCP Authentication",
    "DCP Verification",
    "End-to-End TSTV",
    "ECC/ECDH + HKDF"
]:

    data = master_performance_df[
        master_performance_df["Experiment"] == experiment
    ].copy()

    for _, row in data.iterrows():

        if experiment == "ECC/ECDH + HKDF":
            mean_per_operation = row[
                "Mean Time per User (s)"
            ]
            operation_label = "Session key"
        elif experiment == "Registration":
            mean_per_operation = row[
                "Mean Time per User (s)"
            ]
            operation_label = "User"
        else:
            mean_per_operation = row[
                "Mean Time per Transaction (s)"
            ]
            operation_label = "Transaction"

        publication_rows.append({
            "Experiment": experiment,
            "Users": int(row["Users"]),
            "Trials": int(row["Trials"]),
            "Mean Time (s)": row["Mean Total Time (s)"],
            "SD (s)": row["Between-Trial SD (s)"],
            "95% CI": (
                f"[{row['95% CI Lower (s)']:.6f}, "
                f"{row['95% CI Upper (s)']:.6f}]"
            ),
            "Mean Time per Operation (s)": mean_per_operation,
            "Operation": operation_label,
            "Success Rate (%)": (
                row["Verification Rate"] * 100
            )
        })

publication_table_df = pd.DataFrame(
    publication_rows
)

display(
    publication_table_df.round({
        "Mean Time (s)": 6,
        "SD (s)": 6,
        "Mean Time per Operation (s)": 9,
        "Success Rate (%)": 2
    })
)

print(
    "\nPublication-ready performance table created."
)

### 14.2 Reproducibility environment

In [ ]:
# Cell 56: Reproducibility / execution environment

import sys
import platform
import os
import subprocess
import psutil
import numpy as np
import pandas as pd

print("===== TSTV EXPERIMENTAL ENVIRONMENT =====")

print("Python version:")
print(sys.version)

print("\nPlatform:")
print(platform.platform())

print("\nProcessor:")
print(platform.processor())

print("\nMachine:")
print(platform.machine())

print("\nCPU cores:")
print(os.cpu_count())

print("\nMemory:")
print(f"{psutil.virtual_memory().total / (1024**3):.2f} GB")

print("\nNumPy version:")
print(np.__version__)

print("Pandas version:")
print(pd.__version__)

try:
    import cryptography
    print("Cryptography version:")
    print(cryptography.__version__)
except Exception as e:
    print("Cryptography version: unavailable")

print("\nRandom seed:")
print(SEED)

print("\nUser-load levels:")
print(USER_SIZES)

print("\nTrials per workload:")
print(10)

print("\n========================================")

### 14.3 Final end-to-end performance table

In [ ]:
# Cell 57: Final end-to-end TSTV publication table

e2e_tstv_df = (
    master_performance_df[
        master_performance_df["Experiment"] == "End-to-End TSTV"
    ]
    .copy()
    .sort_values("Users")
)

final_tstv_table = pd.DataFrame({
    "Users": e2e_tstv_df["Users"].astype(int),
    "Trials": e2e_tstv_df["Trials"].astype(int),
    "Mean total time (s)": e2e_tstv_df["Mean Total Time (s)"],
    "SD (s)": e2e_tstv_df["Between-Trial SD (s)"],
    "95% CI lower (s)": e2e_tstv_df["95% CI Lower (s)"],
    "95% CI upper (s)": e2e_tstv_df["95% CI Upper (s)"],
    "Mean time/transaction (ms)": (
        e2e_tstv_df["Mean Time per Transaction (s)"] * 1000
    ),
    "Verification rate (%)": (
        e2e_tstv_df["Verification Rate"] * 100
    )
})

display(
    final_tstv_table.round({
        "Mean total time (s)": 6,
        "SD (s)": 6,
        "95% CI lower (s)": 6,
        "95% CI upper (s)": 6,
        "Mean time/transaction (ms)": 3,
        "Verification rate (%)": 1
    })
)

print("\nFinal end-to-end TSTV table created.")

### 14.4 End-to-end performance figure

In [ ]:
# Cell 58: End-to-end TSTV performance vs. user load
# Mean transaction time with 95% confidence intervals

import matplotlib.pyplot as plt

plot_df = final_tstv_table.copy()

x = plot_df["Users"].to_numpy()
y = plot_df["Mean time/transaction (ms)"].to_numpy()

# Convert total-time CI into per-transaction CI in milliseconds
ci_lower = (
    plot_df["95% CI lower (s)"].to_numpy()
    / plot_df["Users"].to_numpy()
    * 1000
)

ci_upper = (
    plot_df["95% CI upper (s)"].to_numpy()
    / plot_df["Users"].to_numpy()
    * 1000
)

yerr_lower = y - ci_lower
yerr_upper = ci_upper - y

plt.figure(figsize=(7, 4.5))

plt.errorbar(
    x,
    y,
    yerr=[yerr_lower, yerr_upper],
    fmt="o-",
    capsize=4,
    linewidth=1.5,
    markersize=5
)

plt.xlabel("Number of users")
plt.ylabel("Mean end-to-end time per transaction (ms)")
plt.xticks(x)
plt.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

print("Figure 1 generated: End-to-end TSTV performance with 95% CI.")

### 14.5 Payload-size figure

In [ ]:
# Cell 59: Publication-ready payload-size figure

import numpy as np
import matplotlib.pyplot as plt

payload_plot_df = payload_summary_df.sort_values(
    "Payload Size (bytes)"
).copy()

payload_sizes = payload_plot_df["Payload Size (bytes)"].to_numpy()
means_ms = payload_plot_df["Mean Time (s)"].to_numpy() * 1000
sd_ms = payload_plot_df["Std Dev (s)"].to_numpy() * 1000

# 95% CI for 10 trials: t(0.975, 9) = 2.262157
tcrit = 2.262157
margin_ms = tcrit * sd_ms / np.sqrt(
    payload_plot_df["Trials"].to_numpy()
)

x = np.arange(len(payload_sizes))

plt.figure(figsize=(7, 4.5))

plt.errorbar(
    x,
    means_ms,
    yerr=margin_ms,
    fmt="o-",
    capsize=4,
    linewidth=1.5,
    markersize=5
)

plt.xlabel("Payload size (bytes)")
plt.ylabel("Mean end-to-end processing time (ms)")
plt.xticks(x, payload_sizes)
plt.grid(True, alpha=0.25)

plt.tight_layout()

plt.savefig(
    "TSTV_Figure_2_Payload_Size_Performance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Figure 2 generated: payload-size performance "
    "with 95% confidence intervals."
)

## 15. Final security-validation table

In [ ]:
# Cell 60: Final security validation table
#
# These are the eight attack scenarios actually implemented
# and tested in the TSTV prototype.

security_validation_df = pd.DataFrame([
    {
        "Attack scenario": "Invalid credentials",
        "Stage": "UCP",
        "Security check": "Credential verification",
        "Observed result": "Rejected - Credential mismatch",
        "Detected": True
    },
    {
        "Attack scenario": "Unauthorized user",
        "Stage": "UCP",
        "Security check": "Authorization",
        "Observed result": "Rejected - Unauthorized user",
        "Detected": True
    },
    {
        "Attack scenario": "Invalid source",
        "Stage": "DCP",
        "Security check": "Source verification",
        "Observed result": "DCP Rejected",
        "Detected": True
    },
    {
        "Attack scenario": "Invalid destination",
        "Stage": "DCP",
        "Security check": "Destination verification",
        "Observed result": "DCP Rejected",
        "Detected": True
    },
    {
        "Attack scenario": "Tampered encrypted data",
        "Stage": "DCP",
        "Security check": "HMAC verification",
        "Observed result": "DCP Rejected",
        "Detected": True
    },
    {
        "Attack scenario": "Forged/modified HMAC",
        "Stage": "DCP",
        "Security check": "HMAC verification",
        "Observed result": "DCP Rejected",
        "Detected": True
    },
    {
        "Attack scenario": "Invalid data flag",
        "Stage": "DCP",
        "Security check": "Data-flag verification",
        "Observed result": "DCP Rejected",
        "Detected": True
    },
    {
        "Attack scenario": "Expired TTL",
        "Stage": "DCP",
        "Security check": "TTL/timestamp verification",
        "Observed result": "DCP Rejected",
        "Detected": True
    }
])

security_validation_df["Detected (%)"] = (
    security_validation_df["Detected"].astype(int) * 100
)

display(security_validation_df)

total_attacks = len(security_validation_df)
detected_attacks = security_validation_df["Detected"].sum()

print("\n===== SECURITY VALIDATION SUMMARY =====")
print(f"Attack scenarios tested: {total_attacks}")
print(f"Attack scenarios detected: {detected_attacks}")
print(
    f"Detection rate: "
    f"{detected_attacks / total_attacks * 100:.1f}%"
)
print("=======================================")

## 16. Attack-load sensitivity analysis

In [ ]:
attack_loads = [0.10, 0.20, 0.30]
user_sizes = [100, 200, 300, 400, 500]

attack_sensitivity_rows = []

for attack_fraction in attack_loads:
    for users in user_sizes:

        result = run_balanced_attack_experiment(
            number_of_users=users,
            attack_ratio=attack_fraction
        )

        attacks = result[result["traffic_type"] == "Attack"]
        legitimate = result[result["traffic_type"] == "Legitimate"]

        detected_attacks = int(
            (attacks["tstv_status"] == "Rejected").sum()
        )

        false_negatives = int(
            (attacks["tstv_status"] != "Rejected").sum()
        )

        false_positives = int(
            (legitimate["tstv_status"] == "Rejected").sum()
        )

        attack_cases = len(attacks)
        legitimate_cases = len(legitimate)

        attack_detection_rate = (
            detected_attacks / attack_cases * 100
            if attack_cases > 0 else 100
        )

        legitimate_acceptance_rate = (
            (legitimate_cases - false_positives)
            / legitimate_cases * 100
            if legitimate_cases > 0 else 100
        )

        attack_sensitivity_rows.append({
            "Users": users,
            "Attack load (%)": attack_fraction * 100,
            "Legitimate cases": legitimate_cases,
            "Attack cases": attack_cases,
            "Detected attacks": detected_attacks,
            "Attack detection rate (%)": attack_detection_rate,
            "False positives": false_positives,
            "False negatives": false_negatives,
            "Legitimate acceptance rate (%)":
                legitimate_acceptance_rate
        })

attack_sensitivity_df = pd.DataFrame(
    attack_sensitivity_rows
)

display(attack_sensitivity_df)

total_conditions = len(attack_sensitivity_df)
total_attack_cases = int(
    attack_sensitivity_df["Attack cases"].sum()
)
total_detected_attacks = int(
    attack_sensitivity_df["Detected attacks"].sum()
)
total_false_positives = int(
    attack_sensitivity_df["False positives"].sum()
)
total_false_negatives = int(
    attack_sensitivity_df["False negatives"].sum()
)

overall_detection_rate = (
    total_detected_attacks /
    total_attack_cases *
    100
)

print("\n===== ATTACK-LOAD SENSITIVITY SUMMARY =====")
print("Total workload conditions:", total_conditions)
print("Total attack cases:", total_attack_cases)
print("Detected attack cases:", total_detected_attacks)
print(
    "Overall attack detection rate:",
    f"{overall_detection_rate:.1f}%"
)
print("Total false positives:", total_false_positives)
print("Total false negatives:", total_false_negatives)
print("============================================")

## 17. Stage-wise performance comparison

In [ ]:
# Cell 62: Final stage-wise TSTV performance comparison

stage_experiments = [
    "Registration",
    "UCP Authentication",
    "DCP Verification",
    "End-to-End TSTV"
]

stage_rows = []

for experiment in stage_experiments:

    data = master_performance_df[
        master_performance_df["Experiment"] == experiment
    ].copy().sort_values("Users")

    for _, row in data.iterrows():

        users = int(row["Users"])

        # Normalize total workload time to time per operation.
        # Each workload contains one operation per user.
        time_per_operation_ms = (
            row["Mean Total Time (s)"] / users * 1000
        )

        stage_rows.append({
            "Stage": experiment,
            "Users": users,
            "Mean total time (s)":
                row["Mean Total Time (s)"],
            "SD (s)":
                row["Between-Trial SD (s)"],
            "95% CI lower (s)":
                row["95% CI Lower (s)"],
            "95% CI upper (s)":
                row["95% CI Upper (s)"],
            "Mean time per operation (ms)":
                time_per_operation_ms,
            "Success rate (%)":
                row["Verification Rate"] * 100
        })

stage_performance_df = pd.DataFrame(stage_rows)

display(
    stage_performance_df.round({
        "Mean total time (s)": 6,
        "SD (s)": 6,
        "95% CI lower (s)": 6,
        "95% CI upper (s)": 6,
        "Mean time per operation (ms)": 3,
        "Success rate (%)": 1
    })
)

print("\n===== STAGE-WISE PERFORMANCE SUMMARY =====")

for stage in stage_experiments:

    subset = stage_performance_df[
        stage_performance_df["Stage"] == stage
    ]

    min_time = subset[
        "Mean time per operation (ms)"
    ].min()

    max_time = subset[
        "Mean time per operation (ms)"
    ].max()

    print(
        f"{stage}: "
        f"{min_time:.3f}–{max_time:.3f} ms/operation"
    )

print("==========================================")

### 17.1 Stage-wise performance figure

In [ ]:
# Cell 63: Stage-wise TSTV performance figure

import matplotlib.pyplot as plt

stage_order = [
    "Registration",
    "UCP Authentication",
    "DCP Verification",
    "End-to-End TSTV"
]

plt.figure(figsize=(8, 5))

for stage in stage_order:

    subset = stage_performance_df[
        stage_performance_df["Stage"] == stage
    ].sort_values("Users")

    plt.plot(
        subset["Users"],
        subset["Mean time per operation (ms)"],
        marker="o",
        linewidth=1.5,
        markersize=5,
        label=stage
    )

plt.xlabel("Number of users")
plt.ylabel("Mean processing time per operation (ms)")
plt.xticks([100, 200, 300, 400, 500])

# Log scale prevents the small UCP/DCP values from being
# visually compressed by the much larger end-to-end values.
plt.yscale("log")

plt.grid(True, which="both", alpha=0.25)
plt.legend()

plt.tight_layout()

plt.savefig(
    "TSTV_Figure_3_Stage_Performance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Figure 3 generated: Stage-wise TSTV performance.")